# FiCO₂ ⇌ PaCO₂ Conversion
### A simplified, ventilation-aware converter for CO₂ challenges

*Supplementary tool for the isometabolism review (BOLD-MRI / CMRO₂ during hypercapnia).*

CO₂ challenges are reported in two incompatible currencies: some studies specify the
**inspired fraction** (FiCO₂, e.g. "5% CO₂") and others a **partial pressure**
(PaCO₂ ≈ PetCO₂, in mmHg or kPa). This notebook converts between them **in both
directions**, accounting for the fact that raising inspired CO₂ raises PaCO₂ *but the
resulting hypercapnia stimulates ventilation*, which offloads part of the added CO₂.

See `review_derivation` (simplified, fixed-slope form) and `technical_note`
(reversible, multi-unit, adjustable slope) for the full derivation.

## 1. The governing equation

Alveolar CO₂ mass balance **with inspired CO₂** (Slessarev et al., 2007), converted to
partial pressures by Dalton's law:

$$P_aCO_2 = \underbrace{F_iCO_2\,(P_{atm}-P_{H_2O})}_{P_iCO_2} + \frac{K\,\dot V_{CO_2}}{\dot V_A(P_aCO_2)} \qquad (1)$$

with **CO₂-sensitive** alveolar ventilation (the hypercapnic ventilatory response, HCVR):

$$\dot V_A(P_aCO_2) = \dot V_{A,\text{base}} + S\,(P_aCO_2 - P_aCO_{2,\text{base}}) \qquad (2)$$

Because $\dot V_A$ is **linear** in PaCO₂, equation (1) is a **quadratic** in PaCO₂ with an
exact closed-form (physical) root — no iterative solver needed:

$$P_aCO_2 = \frac{(S\,P_iCO_2 - D) + \sqrt{(S\,P_iCO_2 - D)^2 + 4S(P_iCO_2 D + K\dot V_{CO_2})}}{2S},\quad D \equiv \dot V_{A,\text{base}} - S\,P_aCO_{2,\text{base}} \qquad (3)$$

The forward direction (PaCO₂ → FiCO₂) is explicit:

$$F_iCO_2 = \frac{P_aCO_2 - K\dot V_{CO_2}/\dot V_A(P_aCO_2)}{P_{atm}-P_{H_2O}} \qquad (4)$$

## 2. Constants and their sources

| Quantity | Value | Source |
|---|---|---|
| $K$ | 0.863 | STPD→BTPS + fraction→pressure unit constant (West, 2012) |
| $P_{atm}$ | 760 mmHg | sea-level barometric pressure |
| $P_{H_2O}$ | 47 mmHg | saturated vapour pressure at 37 °C |
| $P_{atm}-P_{H_2O}$ | 713 mmHg | dry-gas scaling for Dalton's law |
| $\dot V_{CO_2}$ | 200 mL·min⁻¹ | resting CO₂ output (range 150–200) |
| $P_aCO_{2,\text{base}}$ | 40 mmHg | normocapnic resting baseline |
| $\dot V_{A,\text{base}}$ | ≈ 4.3 L·min⁻¹ | set = $K\dot V_{CO_2}/P_aCO_{2,\text{base}}$ (self-consistent) |
| $\dot V_{A,\text{rest}}$ | **4.2 L·min⁻¹** | literature resting alveolar ventilation (West; Nunn's) — fallback |
| $S$ | 2.69 L·min⁻¹·mmHg⁻¹ | mean HCVR slope (Hirshman et al., 1975; range 1.16–5.95) |

**On $\dot V_{A,\text{base}}$ — two cases:**
- **Baseline PaCO₂ known** → $\dot V_{A,\text{base}} = K\dot V_{CO_2}/P_aCO_{2,\text{base}}$ (self-consistent: FiCO₂=0 returns the baseline exactly).
- **Only FiCO₂ known** → assume $P_aCO_{2,\text{base}}=40$ mmHg and use the literature resting value $\dot V_{A,\text{base}}=4.2$ L·min⁻¹. (The steep HCVR absorbs the tiny mismatch, so FiCO₂=0 → ≈ 40.04 mmHg.)

In [ ]:
from fico2_paco2_converter import (
    Params, params_from_baseline, params_resting_default,
    paco2_to_fico2, fico2_to_paco2, fico2_to_paco2_numeric,
    mmhg_to_kpa, kpa_to_mmhg, VA_REST, DEFAULT_BASELINE,
)

# Quick sanity check: a common '5% CO2' inhalation, no known baseline.
p = params_resting_default()
pa = fico2_to_paco2(5.0, p)
print(f'5% CO2  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.2f} kPa')
print(f'(PaCO2_base={p.PaCO2_base:.0f} mmHg, VA_base={p.VA_base:.1f} L/min, S={p.S})')

## 3. Direction 1 — I have a target PaCO₂ → estimate FiCO₂

Supply the target (hypercapnic) PaCO₂ and the baseline PaCO₂. $\dot V_{A,\text{base}}$ is
derived self-consistently from the baseline.

In [ ]:
# --- edit these two numbers ---
paco2_target = 50.0   # mmHg, the hypercapnic level you want to reach
paco2_base   = 40.0   # mmHg, the subject's normocapnic baseline
# --------------------------------

p = params_from_baseline(paco2_base)
fico2 = paco2_to_fico2(paco2_target, p)
print(f'VA_base           = {p.VA_base:.3f} L/min (self-consistent)')
print(f'VA at hypercapnia = {p.VA(paco2_target):.3f} L/min')
print(f'PaCO2 {paco2_target:.1f} mmHg  ->  FiCO2 = {fico2:.3f} %'      f'   (PICO2 = {fico2/100*p.Pdry:.1f} mmHg)')

## 4. Direction 2 — I have an FiCO₂ → estimate PaCO₂

If you know the baseline PaCO₂, set it below; otherwise leave `paco2_base = None` to use
the literature resting fallback ($P_aCO_{2,\text{base}}=40$, $\dot V_{A,\text{base}}=4.2$ L·min⁻¹).

In [ ]:
# --- edit these ---
fico2_percent = 5.0    # inspired CO2, %
paco2_base    = None   # mmHg, or None if unknown  ->  resting fallback
# -------------------

if paco2_base is None:
    p = params_resting_default()
    tag = f'resting fallback (base={p.PaCO2_base:.0f}, VA_base={p.VA_base:.1f} L/min)'
else:
    p = params_from_baseline(paco2_base)
    tag = f'baseline known (base={p.PaCO2_base:.0f}, VA_base={p.VA_base:.3f} L/min)'

pa = fico2_to_paco2(fico2_percent, p)
print(f'mode: {tag}')
print(f'FiCO2 {fico2_percent:.2f} %  ->  PaCO2 = {pa:.2f} mmHg = {mmhg_to_kpa(pa):.3f} kPa')
print(f'rise from baseline: {pa - p.PaCO2_base:+.2f} mmHg')

## 5. Reference table (technical note, Table 1)

Steady-state PaCO₂ for a fixed inspired CO₂ at sea level, default normocapnic
parameters ($S=2.69$, baseline 40 mmHg).

In [ ]:
p = params_from_baseline(40.0)
print(f"{'FiCO2 %':>8} {'PICO2 mmHg':>11} {'PaCO2 mmHg':>11} {'PaCO2 kPa':>10}")
for fi in (0, 2, 5, 7, 8):
    pa = fico2_to_paco2(fi, p)
    pico2 = fi/100 * p.Pdry
    print(f'{fi:8.0f} {pico2:11.1f} {pa:11.2f} {mmhg_to_kpa(pa):10.2f}')

## 6. Sensitivity to the HCVR slope S

The single largest source of uncertainty is $S$. Hirshman et al. (1975) report a mean of
2.69 with a wide individual range (1.16–5.95). Below, the same FiCO₂ is mapped across that
range so you can see how much the estimated PaCO₂ depends on the assumed slope.

In [ ]:
fico2 = 5.0
print(f'FiCO2 = {fico2}%, baseline 40 mmHg\n')
print(f"{'S':>6} {'PaCO2 mmHg':>11}")
for S in (1.16, 1.96, 2.69, 4.04, 5.95):
    p = params_from_baseline(40.0, S=S)
    print(f'{S:6.2f} {fico2_to_paco2(fico2, p):11.2f}')

In [ ]:
# Optional plot (requires matplotlib):  FiCO2 -> PaCO2 across a range of S.
try:
    import numpy as np, matplotlib.pyplot as plt
    fico2_grid = np.linspace(0, 8, 81)
    plt.figure(figsize=(6, 4))
    for S in (1.16, 2.69, 5.95):
        p = params_from_baseline(40.0, S=S)
        pa = [fico2_to_paco2(f, p) for f in fico2_grid]
        plt.plot(fico2_grid, pa, label=f'S = {S}')
    plt.xlabel('FiCO$_2$ (%)'); plt.ylabel('PaCO$_2$ (mmHg)')
    plt.title('FiCO$_2$ -> PaCO$_2$ vs HCVR slope'); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed - skipping plot (pip install matplotlib)')

## 7. Interactive prompt (optional)

Run the cell below to be *prompted* for values in the same way as the command-line tool
(`python fico2_paco2_converter.py -i`).

In [ ]:
from fico2_paco2_converter import interactive
interactive()

---
### Assumptions & limitations
- Ideal alveolar–arterial equilibration: PaCO₂ ≈ PACO₂ ≈ PetCO₂ (healthy lungs, no significant V/Q mismatch or shunt).
- HCVR treated as **linear** over the working range (≈ 40–80 mmHg).
- A single **population-mean** slope $S=2.69$; it will not match any individual (see §6).
- Published $S$ values are usually **minute**-ventilation slopes ($\Delta\dot V_E/\Delta P_aCO_2$); we use them for the **alveolar** slope, valid while dead-space ventilation changes little.

Best used to **harmonise reported CO₂-challenge magnitudes across studies**, not to predict an individual subject's PaCO₂.